CREATE THE TRAIN DATASTRUCTURE FOR 3 EDGES

In [1]:
import pandas as pd
import numpy as np

# Specifica le features da mantenere (le stesse di sempre)
features_to_keep = [
    'Source IP', 'Destination IP', 'Source Port', 'Destination Port', 'Protocol', 'Timestamp', 'Label',
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets',
    'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Idle Mean', 'Idle Min', 'Active Mean',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
    'SYN Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward'
]

# Carica il dataset
data = pd.read_csv('./data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
data.columns = data.columns.str.strip()

# Filtra solo le colonne necessarie
available_features = [f for f in features_to_keep if f in data.columns]
filtered_data = data[available_features].copy()

print("Conversione dei Timestamp originali...")
# NESSUNO SHUFFLE! Manteniamo l'ordine cronologico per i grafi
filtered_data['Timestamp'] = pd.to_datetime(filtered_data['Timestamp'], dayfirst=True, format='mixed', errors='coerce')

# Rimuoviamo eventuali righe corrotte
filtered_data = filtered_data.dropna(subset=['Timestamp'])

print("Ordinamento cronologico di tutti i flussi...")
filtered_data = filtered_data.sort_values(by='Timestamp')

filtered_data.to_csv('filtered_all_3edge.csv', index=False)
print("Dati cronologici salvati in 'filtered_all_3edge.csv'.")

Conversione dei Timestamp originali...
Ordinamento cronologico di tutti i flussi...
Dati cronologici salvati in 'filtered_all_3edge.csv'.


CREATED 1-MINUTE GRAPH WITH 3 EDGES FROM TRAIN DATASET

In [2]:
import os
import networkx as nx
import pandas as pd
import pickle
import random

def create_train_graphs_edge_labels(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    # Raggruppiamo per 1 minuto
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='1min'))]
    
    # SHUFFLE DEI MINUTI E SPLIT 80% PER IL TRAIN
    random.seed(123)
    random.shuffle(time_slices)
    split_idx = int(len(time_slices) * 0.8)
    train_slices = time_slices[:split_idx]

    label_mapping = {'BENIGN': 0, 'DDoS': 1}
    count = 0

    for slice_index, slice_df in enumerate(train_slices):
        if slice_df.empty: continue

        G = nx.MultiDiGraph()
        for _, row in slice_df.iterrows():
            src_ip, dst_ip = row['Source IP'], row['Destination IP']
            label = label_mapping.get(row['Label'], 0)

            if pd.isna(src_ip) or pd.isna(dst_ip): continue

            if not G.has_node(src_ip): G.add_node(src_ip)
            if not G.has_node(dst_ip): G.add_node(dst_ip)

            G.add_edge(src_ip, dst_ip, label=label, interaction='network', **{'Source Port': row.get('Source Port', 0), 'Destination Port': row.get('Destination Port', 0)})
            G.add_edge(src_ip, dst_ip, label=label, interaction='context', **{'Flow Duration': row.get('Flow Duration', 0), 'Flow IAT Mean': row.get('Flow IAT Mean', 0)})
            G.add_edge(src_ip, dst_ip, label=label, interaction='knowledge', **{'Packet Length Std': row.get('Packet Length Std', 0), 'SYN Flag Count': row.get('SYN Flag Count', 0)})

        graph_path = os.path.join(output_dir, f"train_graph_{slice_index}.pkl")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f)
        count += 1

    print(f"Creati con successo {count} grafi di TRAIN in '{output_dir}'")

if __name__ == "__main__":
    df_train = pd.read_csv('filtered_all_3edge.csv')
    df_train['Timestamp'] = pd.to_datetime(df_train['Timestamp'])
    df_train = df_train.set_index('Timestamp').sort_index()

    output_dir = "train_1min_graphs"
    create_train_graphs_edge_labels(df_train, output_dir)

Creati con successo 74 grafi di TRAIN in 'train_1min_graphs'


 COMMUNITY DETECTION FOR GRAPHS AND THEN UPDATE THE GRAPH WITH THE LABEL OF COMMUNITY FOR EACH NODE 


In [3]:
def detect_and_label_communities_lpa(graph):
    """
    Perform community detection using the Label Propagation Algorithm (LPA) and label nodes with community IDs.
    Adds 'x' attribute based on the 'community' label.

    Parameters:
        graph (nx.MultiDiGraph): Input graph.

    Returns:
        graph (nx.MultiDiGraph): Updated graph with community labels and 'x' attributes.
    """
    # Convert MultiDiGraph to Graph (undirected graph for LPA)
    undirected_graph = nx.Graph(graph)

    # Perform community detection using LPA
    communities = nx.community.label_propagation_communities(undirected_graph)

    # Assign community labels to nodes and add 'x' attribute
    for community_id, community in enumerate(communities):
        for node in community:
            graph.nodes[node]['community'] = community_id
            # L'ID della community è arbitrario: come feature usiamo la sua dimensione
            graph.nodes[node]['community_size'] = len(community)
            #graph.nodes[node]['x'] = [community_id]  # 'x' is a feature; wrap in a list for PyTorch Geometric compatibility

    return graph


def add_structural_features(graph):
    """
    Aggiunge in_degree, out_degree (contati in flussi, cioè solo archi 'network')
    e betweenness centrality a ogni nodo.
    """
    for node in graph.nodes:
        graph.nodes[node]['in_degree'] = sum(1 for _, _, t in graph.in_edges(node, data='interaction') if t == 'network')
        graph.nodes[node]['out_degree'] = sum(1 for _, _, t in graph.out_edges(node, data='interaction') if t == 'network')

    # La centralità non dipende dalla molteplicità degli archi: basta il DiGraph semplice
    centrality = nx.betweenness_centrality(nx.DiGraph(graph))
    for node, value in centrality.items():
        graph.nodes[node]['centrality'] = value

    return graph


def process_graphs_with_lpa(input_dir, output_dir):
    """
    Detect communities using LPA, update graphs with community labels, and add 'x' attribute.

    Parameters:
        input_dir (str): Directory containing input graphs.
        output_dir (str): Directory to save updated graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Process each graph file in the input directory
    #for graph_file in os.listdir(input_dir):
    #    if not graph_file.endswith('.gpickle'):
    #        continue

        # Load the graph
    #    graph_path = os.path.join(input_dir, graph_file)
    #    G = nx.read_gpickle(graph_path)

        # Detect communities using LPA and label nodes
    #    G = detect_and_label_communities_lpa(G)

        # Save the updated graph
    #    updated_graph_path = os.path.join(output_dir, graph_file)
    #    nx.write_gpickle(G, updated_graph_path)
    #    print(f"Updated graph with LPA communities and 'x' attribute saved to {updated_graph_path}")

    #CAMBIO QUESTA PARTE DI CODICE IN QUANTO ORA ABBIAMO SALVATO IL GRAFO 
    #CON PICKLE:

    # Verifica se ci sono file nella cartella di partenza
    if not os.path.exists(input_dir) or not os.listdir(input_dir):
        print(f"Errore: la cartella di input {input_dir} è vuota o non esiste! Genera prima i grafi orari.")
        return

    count = 0
    for graph_file in os.listdir(input_dir):
        # CORRETTO: Cambiato da .gpickle a .pkl
        if not graph_file.endswith('.pkl'):
            continue

        graph_path = os.path.join(input_dir, graph_file)
        
        # Carica il file usando pickle nativo
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)

        # Calcola le comunità LPA
        G = detect_and_label_communities_lpa(G)
        G = add_structural_features(G)

        # Salva il grafo aggiornato usando pickle nativo nella cartella _commun
        updated_graph_path = os.path.join(output_dir, graph_file)
        with open(updated_graph_path, 'wb') as f:
            pickle.dump(G, f)
            
        print(f"Updated graph with LPA saved to {updated_graph_path}")
        count += 1

    print(f"Elaborazione completata! Elaborati {count} file.")


# Example usage
if __name__ == "__main__":
    # Puntiamo alle nuove cartelle
    input_graph_dir = "train_1min_graphs"
    output_graph_dir = "train_commun_graphs"

    process_graphs_with_lpa(input_graph_dir, output_graph_dir)


Updated graph with LPA saved to train_commun_graphs\train_graph_0.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_1.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_10.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_11.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_12.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_13.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_14.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_15.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_16.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_17.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_18.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_19.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_2.pkl
Updated graph with LPA saved to train_commun_graphs\train_graph_20.pkl
Updated g

CONVERT MULTIPGRAPH TO HETERODATA 

In [4]:
import torch
from torch_geometric.data import HeteroData
import pickle
import networkx as nx

def multiDiGraph_to_hetero_with_label(G: nx.MultiDiGraph) -> HeteroData:
    """
    Converts a MultiDiGraph with multiple edge types to a HeteroData object.
    Preserves the 'label' field in data[rel_type].edge_label.
    """
    data = HeteroData()
    node_mapping = {node: i for i, node in enumerate(G.nodes())}
    data['ip'].num_nodes = G.number_of_nodes()

    # Add node-level features
    x = []
    community_labels = []

    for node in G.nodes():
        community_size = G.nodes[node].get('community_size', 0)
        in_deg = G.nodes[node].get('in_degree', 0)
        out_deg = G.nodes[node].get('out_degree', 0)
        centrality = G.nodes[node].get('centrality', 0)

        # Il vettore delle feature dei nodi avrà ora 4 dimensioni (sia in_
        # degree che out_degree, community e centrality) e non 3 come nel paper
        x.append([float(community_size), float(in_deg), float(out_deg), float(centrality)])

    data['ip'].x = torch.tensor(x, dtype=torch.float)

    # Process each edge from G.
    for u, v, key, edge_attrs in G.edges(data=True, keys=True):
        src = node_mapping[u]
        dst = node_mapping[v]
        # Il tipo di relazione è nell'attributo 'interaction' (la key è ora un intero progressivo)
        key = edge_attrs['interaction']
        rel_type = ('ip', key, 'ip')

        if rel_type not in data.edge_types:
            data[rel_type].edge_index = []
            data[rel_type].edge_attr = []
            data[rel_type].edge_label = []  # Container for the label

        data[rel_type].edge_index.append([src, dst])
        feature_vec = []

        #if key == 'network':
        #    for attr_name in [ #selected fetures]:
        #        feature_vec.append(edge_attrs.get(attr_name, 0))
        #elif key == 'context':
        #    for attr_name in [ #selected fetures]:
        #        feature_vec.append(edge_attrs.get(attr_name, 0))
        #elif key == 'knowledge':
        #    for attr_name in [ #selected fetures]:
        #        feature_vec.append(edge_attrs.get(attr_name, 0))

        if key == 'network': # Network Communication Features 
            feature_vec = [edge_attrs.get('Source Port', 0), edge_attrs.get('Destination Port', 0)]
        elif key == 'context': # Contextual Features (IAT, temporal)
            feature_vec = [edge_attrs.get('Flow Duration', 0), edge_attrs.get('Flow IAT Mean', 0)]
        elif key == 'knowledge': # Knowledge-Based (Flags, Packet size) 
            feature_vec = [edge_attrs.get('Packet Length Std', 0), edge_attrs.get('SYN Flag Count', 0)]

        data[rel_type].edge_attr.append(feature_vec)
        # Save the label
        label = edge_attrs.get('label', 0)  # Default to -1 if label is missing
        #if label == -1:
        #    print(f"Warning: Missing or invalid label for edge {u} -> {v} of type {key}")
        data[rel_type].edge_label.append(label)

    # Convert lists to tensors.
    for rel_type in data.edge_types:
        data[rel_type].edge_index = torch.tensor(data[rel_type].edge_index, dtype=torch.long).t().contiguous()
        if data[rel_type].edge_attr:
            data[rel_type].edge_attr = torch.tensor(data[rel_type].edge_attr, dtype=torch.float)
        if data[rel_type].edge_label:
            data[rel_type].edge_label = torch.tensor(data[rel_type].edge_label, dtype=torch.long)
    return data

def process_and_save_hetero_graphs_with_label(input_dir, output_dir):
    """
    Converts all .gpickle graphs in a directory to HeteroData objects and saves them as .pt,
    preserving the 'label' field in data[rel_type].edge_label.
    """
    os.makedirs(output_dir, exist_ok=True)
    for graph_file in os.listdir(input_dir):
        #if not graph_file.endswith('.gpickle'):
        if not graph_file.endswith('.pkl'):
            continue

        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)

        #G = nx.read_gpickle(graph_path)
        hetero_data = multiDiGraph_to_hetero_with_label(G)
        hetero_path = os.path.join(output_dir, graph_file.replace('.pkl', '.pt'))
        torch.save(hetero_data, hetero_path)

        #print(f"Saved HeteroData with labels to {hetero_path}")
        print(f"Converted: {graph_file} -> {hetero_path}")



if __name__ == "__main__":
    # Puntiamo alle nuove cartelle
    input_dir_pkl = "train_commun_graphs"
    output_dir_pt = "train_hetero_graphs_pt"
    
    process_and_save_hetero_graphs_with_label(input_dir_pkl, output_dir_pt)

c:\Users\irene\Documents\SAPIENZA\TESI_GraphIDS\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Converted: train_graph_0.pkl -> train_hetero_graphs_pt\train_graph_0.pt
Converted: train_graph_1.pkl -> train_hetero_graphs_pt\train_graph_1.pt
Converted: train_graph_10.pkl -> train_hetero_graphs_pt\train_graph_10.pt
Converted: train_graph_11.pkl -> train_hetero_graphs_pt\train_graph_11.pt
Converted: train_graph_12.pkl -> train_hetero_graphs_pt\train_graph_12.pt
Converted: train_graph_13.pkl -> train_hetero_graphs_pt\train_graph_13.pt
Converted: train_graph_14.pkl -> train_hetero_graphs_pt\train_graph_14.pt
Converted: train_graph_15.pkl -> train_hetero_graphs_pt\train_graph_15.pt
Converted: train_graph_16.pkl -> train_hetero_graphs_pt\train_graph_16.pt
Converted: train_graph_17.pkl -> train_hetero_graphs_pt\train_graph_17.pt
Converted: train_graph_18.pkl -> train_hetero_graphs_pt\train_graph_18.pt
Converted: train_graph_19.pkl -> train_hetero_graphs_pt\train_graph_19.pt
Converted: train_graph_2.pkl -> train_hetero_graphs_pt\train_graph_2.pt
Converted: train_graph_20.pkl -> train_heter